# BT4012 Kaggle: detecting illicit Bitcoin transactions (e1122376)

**The task.** Every row is a Bitcoin transaction described by 165 anonymised features (the Elliptic data set). The target says
whether it is illicit (11.67% of the labelled rows) or licit. The split is by time: `train.csv` covers days 1-35 and `test.csv`
holds the 15,329 labelled transactions of days 36-49. The score is ROC AUC, so only the order of the predictions matters.
A submission is `index,target` in the order of `sample_submission.csv`.

**The final model** is `submissions/blend_tabpfn_kumo22_55.csv` (public leaderboard 0.97357): a rank average of two pretrained
tabular foundation models that read labelled rows as context and score the test rows without any training on this data.

* **TabPFN v3.5** (weight .45): 11 bags with different seeds, first days and context sizes.
* **NVIDIA Kumo Tabular, large** (weight .55): 22 runs with different seeds, first days and context sizes.

It uses the 165 supplied features unchanged. Only the competition's `train.csv`, `test.csv`, `sample_submission.csv` and
`txs_edgelist.csv` are read. No outside rows or labels are used. The pretrained weights of the two models are an outside
ingredient and are described in section 4.3, together with a note on the project's literature reading.

**Sections:** 1 setup, 2 data loading and preprocessing checks, 3 feature engineering (evidence for using none),
4 the model, 5 prediction validation (time-based holdouts, run live on the GPU), 6 test predictions and the submission file,
7 checks.

**What the two run modes do, and do not, show.** `RERUN` (section 1, from the environment variable `E1122376_RERUN`) decides how
section 6 gets the 4 TabPFN files and 22 Kumo files:

| Mode | How to select | What section 6 does | Time |
|---|---|---|---|
| Fast path (default) | nothing | loads the committed per-run prediction files from `submissions/` (made earlier by `src/model_tabpfn.py` and `src/fm/run_kumo.py` from `train.csv` and `test.csv`), rebuilds the three blends and asserts they equal the committed blends exactly | seconds |
| Full rerun | `$env:E1122376_RERUN = "1"` | recomputes every TabPFN bag and Kumo run from the raw data into `rebuild/rerun/`, then compares each file with the committed one | about 2.5 hours on one RTX 3060 Ti |

The fast path therefore checks the **blending arithmetic and the submission format**, not the models: the exact match in section 7 passes
by construction. Only the full rerun regenerates the predictions from the competition data, and it is **not bit-identical** to the committed files
(several committed bags were run on a CPU and GPU kernels are not deterministic), so it is judged by rank correlation.
The building blocks of the rerun were run at full size and agree (two Kumo runs reproduce the committed files byte for byte, a 12k TabPFN
bag has Spearman 0.992 with the committed bag), but the complete 2.5-hour rerun was **not executed end to end**.

Section 5 (validation) runs both models live on the GPU in both modes. Its predictions are cached in `cache/notebook/validation/`,
so the first execution takes about 10 minutes of GPU time and later ones take seconds. To recompute, point `E1122376_VALID_DIR` at an empty folder
(or delete the cache folder). `$env:E1122376_SKIP_VALIDATION = "1"` skips section 5, so the fast path then needs no GPU.

**How to run.** From the repository root, with the repository's `.venv` (Python 3.11, `requirements.txt`):

```
.venv\Scripts\python.exe -m nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=3600 e1122376.ipynb
```

Do not use `jupyter nbconvert` or `python -m jupyter`: they start the system Python, which lacks xgboost. In the Jupyter interface select the `.venv` kernel.
A full rerun needs `--ExecutePreprocessor.timeout=-1` (the timeout is per cell and the rerun cell takes hours) and should write to a separate file:
`$env:E1122376_RERUN = "1"; .venv\Scripts\python.exe -m nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 --output-dir rebuild --output e1122376_rerun.ipynb e1122376.ipynb`.
Kumo needs a newer PyTorch than TabPFN, so it runs in a second environment, `.venv-fm` (`requirements-fm.txt`), through a small script that section 4 writes to disk.
Needs for the validation and the full rerun: an NVIDIA GPU with 8 GB (close other GPU programs, the largest runs use almost all of it),
a Prior Labs token in `.env` (`TABPFN_TOKEN`), `.venv-fm`, and internet access for the first download of the weights.

**AI assistance.** This project, including this notebook, `README.md` and `reports/progress.md`, was built with an AI coding assistant
(Claude Code, by Anthropic). The assistant wrote and ran code and experiments and drafted the text. The README repeats this statement.


## 1. Setup and configuration

The imports, the `RERUN` flag, the folders and the secret handling. The TabPFN token is read from `.env` into the
process environment and is never printed.

In [1]:
import gc, json, os, subprocess, sys, time, warnings
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score
try:
    from xgboost import XGBClassifier
except ImportError as err:
    raise ImportError("xgboost is missing, so this is the wrong Python. Run the notebook with the repository's .venv "
                      "(.venv\\Scripts\\python.exe -m nbconvert ... or select the .venv kernel), not with the system Python.") from err

warnings.filterwarnings("ignore")

# RERUN = False (default): load the committed per-run prediction files and rebuild the blends (fast).
# RERUN = True: recompute every TabPFN bag and Kumo run from the raw data (about 2.5 hours on one GPU).
# Switch with  $env:E1122376_RERUN = "1"  in the shell that starts Jupyter or nbconvert.
RERUN = os.environ.get("E1122376_RERUN") == "1"
# E1122376_SKIP_VALIDATION=1 skips section 5 (the live GPU validation), so the fast path then needs no GPU.
SKIP_VALIDATION = os.environ.get("E1122376_SKIP_VALIDATION") == "1"

# Folders. Run the notebook from the repository root.
REPO = Path.cwd()
if not (REPO / "data" / "train.csv").exists():
    raise FileNotFoundError("Start the notebook from the repository root (the folder that holds data/train.csv).")
DATA, SUB = REPO / "data", REPO / "submissions"
WORK = REPO / "cache" / "notebook"        # arrays handed to the Kumo script, plus the validation cache
VALID = REPO / os.environ.get("E1122376_VALID_DIR", str(WORK / "validation"))   # validation cache; point it at an empty folder to recompute
OUT = REPO / "rebuild" / ("rerun" if RERUN else "fastpath")   # everything this notebook writes (gitignored)
for folder in (WORK, VALID, OUT):
    folder.mkdir(parents=True, exist_ok=True)
KUMO_PY = REPO / ".venv-fm" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")

# Secret: TabPFN needs an account token. Read it from .env into the environment, never print it.
env_file = REPO / ".env"
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, value = line.split("=", 1)
            if key.strip() == "TABPFN_TOKEN":
                os.environ.setdefault("TABPFN_TOKEN", value.strip().strip("'\""))

def rel(path):
    return path.relative_to(REPO) if REPO in path.parents else path

print("mode:", "RERUN (full GPU recompute into rebuild/rerun)" if RERUN else "fast path (committed per-run files)",
      "| validation:", "skipped" if SKIP_VALIDATION else f"live, cache {rel(VALID)}")
print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| torch", torch.__version__, "| tabpfn", version("tabpfn"), "| xgboost", version("xgboost"))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      "| TabPFN token set:", "TABPFN_TOKEN" in os.environ, "| Kumo environment found:", KUMO_PY.exists())

mode: RERUN (full GPU recompute into rebuild/rerun) | validation: live, cache cache\notebook\validation
python 3.11.2 | numpy 1.26.4 | pandas 2.2.2 | torch 2.5.1+cu124 | tabpfn 9.2.0 | xgboost 3.2.0
GPU: NVIDIA GeForce RTX 3060 Ti | TabPFN token set: True | Kumo environment found: True


**The recipe of the final model, written down as data.** A TabPFN file with several bags is the mean percentile rank of its bags; the two single-bag files hold the raw P(illicit) of their bag,
as committed (the blend re-ranks every file, so this makes no difference to the result).
Each Kumo file is one run. The weights are the ones used for the submission (README, "Reproducing the submission").
They are kept as strings so they parse to exactly the same floats as in `src/blend.py`.

In [2]:
FEATURES = [f"feat_{i}" for i in range(1, 166)]

# TabPFN: (file, first labelled day in the context, context rows, bag numbers, weight in the TabPFN blend)
TABPFN_RUNS = [
    ("tabpfn_all_bags0-5.csv",          1,  8000, range(6), "0.167"),
    ("tabpfn_ctx12k_bag0.csv",          1, 12000, range(1), "0.083"),
    ("tabpfn_recent18_bags0-2.csv",    18,  8000, range(3), "0.40"),
    ("tabpfn_recent18ctx12k_bag0.csv", 18, 12000, range(1), "0.35"),
]
# Kumo: (file, first labelled day in the context, context rows, seed, weight in the Kumo blend)
# 14380 and 31235 are all the labelled rows from day 18 and from day 1: the "full context" runs.
KUMO_RUNS = (
    [(f"kumo_recent18_s{s}.csv", 18, 8000, s, "0.06818") for s in range(10)]
    + [("kumo_recent18full_s0.csv", 18, 14380, 0, "0.06818")]
    + [(f"kumo_all12k_s{s}.csv", 1, 12000, s, "0.02273") for s in range(10)]
    + [("kumo_allfull_s0.csv", 1, 31235, 0, "0.02273")]
)
# Final blend: TabPFN blend .45 + Kumo blend .55 (public 0.97357). Kumo weights .25 / .33 / .40 / .55 scored 0.97162 / 0.97229 / 0.97280 / 0.97357 (section 5)
FINAL_PARTS = [("blend_tabpfn_recent75_12k.csv", "0.45"), ("kumo22_blend.csv", "0.55")]

print(len(TABPFN_RUNS), "TabPFN files (", sum(len(r[3]) for r in TABPFN_RUNS), "bags ) and", len(KUMO_RUNS), "Kumo runs")

4 TabPFN files ( 11 bags ) and 22 Kumo runs


## 2. Data loading and preprocessing checks

`train.csv` holds every transaction of days 1-35, labelled or not. `test.csv` holds only the labelled transactions of
days 36-49, plus an `index` column that gives the submission order. The loader reads the raw CSV files and stores the
features as float32 and the day as int16, the same casting as `src/data.py`, which the committed prediction files were built with.
Loading the files takes a few seconds.

In [3]:
def read_csv_f32(path):
    """Read a competition CSV; features as float32, day as int16 (same casting as src/data.py)."""
    frame = pd.read_csv(path)
    frame[FEATURES] = frame[FEATURES].astype("float32")
    frame["time_step"] = frame["time_step"].astype("int16")
    frame["txId"] = frame["txId"].astype("int64")
    return frame

train = read_csv_f32(DATA / "train.csv")
test = read_csv_f32(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")

# Labelled training rows: the only rows a model may learn from (110,537 training rows have no label).
labelled = train[train["label"].notna()].copy()
labelled["label"] = labelled["label"].astype("int8")
labelled = labelled.reset_index(drop=True)

print("train", train.shape, "| labelled", len(labelled), "| test", test.shape, "| sample_submission", sample.shape)

train (141772, 168) | labelled 31235 | test (15329, 168) | sample_submission (15329, 2)


**Checks.** The results match the data inspection in `reports/progress.md` section 5. The asserts stop the notebook if the data
ever differs from what the rest of the notebook assumes.

In [4]:
n_ill = int((labelled["label"] == 1).sum())
n_lic = int((labelled["label"] == 0).sum())
checks = pd.Series({
    "missing values in the 165 features (train)": int(train[FEATURES].isna().sum().sum()),
    "missing values in the 165 features (test)": int(test[FEATURES].isna().sum().sum()),
    "training rows without a label": int(train["label"].isna().sum()),
    "labelled rows (licit / illicit)": f"{n_lic:,} / {n_ill:,}",
    "illicit share of labelled rows": f"{n_ill / len(labelled):.2%}",
    "duplicate transaction ids (train + test)": int(pd.concat([train["txId"], test["txId"]]).duplicated().sum()),
    "duplicate feature rows in train": int(train[FEATURES].duplicated().sum()),
    "constant features": int((train[FEATURES].nunique() <= 1).sum()),
    "train days": f"{train['time_step'].min()}-{train['time_step'].max()}",
    "test days": f"{test['time_step'].min()}-{test['time_step'].max()}",
    "test.csv order equals sample_submission order": bool((test["index"].to_numpy() == sample["index"].to_numpy()).all()),
})
print(checks.to_string())

assert checks.iloc[0] == 0 and checks.iloc[1] == 0
assert (n_lic, n_ill) == (27591, 3644) and len(test) == len(sample) == 15329
assert train["time_step"].max() < test["time_step"].min()          # a clean split in time
# The blends add ranks position by position, so test.csv and the submission must list rows in the same order.
assert (test["index"].to_numpy() == sample["index"].to_numpy()).all()

missing values in the 165 features (train)                    0
missing values in the 165 features (test)                     0
training rows without a label                            110537
labelled rows (licit / illicit)                  27,591 / 3,644
illicit share of labelled rows                           11.67%
duplicate transaction ids (train + test)                      0
duplicate feature rows in train                              31
constant features                                             0
train days                                                 1-35
test days                                                 36-49
test.csv order equals sample_submission order              True


**What this means for preprocessing.** Nothing needs cleaning: no missing feature values, no duplicate transactions, no constant
columns. Class balance is handled in the context recipe (section 4), not by reweighting or resampling the data.
No scaling, log transform or outlier clipping is applied:

* Tree models only compare a feature with a threshold, so any order-preserving transform leaves them unchanged (progress.md section 5).
* TabPFN builds its own transformed views of the data inside each estimator. I did not test scaling on TabPFN or Kumo, so the raw values are used.
* Scaling mattered for one model, a neural net (0.9400 with a quantile transform, 0.8748 with a standard scaler, progress.md section 6),
  and it is not part of the final model.
* The 110,537 unlabelled training rows are not used. Pseudo-labelling them (10 variants) never beat a tie and cost up to 0.093 on the
  leaderboard; treating them as clean negatives was also worse in an offline test, about -0.03 (progress.md sections 7 and 16).

The arrays below are what sections 4 to 6 use. They are also saved to `cache/notebook/` for the Kumo script, which runs in another environment.


In [5]:
lab_X = labelled[FEATURES].to_numpy()          # float32, 31,235 x 165
lab_y = labelled["label"].to_numpy()           # 0 = licit, 1 = illicit
lab_day = labelled["time_step"].to_numpy()
test_X = test[FEATURES].to_numpy()
test_index = test["index"].to_numpy()

np.savez(WORK / "labelled.npz", X=lab_X, y=lab_y, day=lab_day)
np.save(WORK / "test_X.npy", test_X)
np.save(WORK / "test_index.npy", test_index)

# The regime break around day 23 (progress section 4.1): the illicit share per block of days.
blocks = pd.cut(lab_day, [0, 22, 35], labels=["days 1-22", "days 23-35"])
print(pd.DataFrame({"rows": pd.Series(lab_y).groupby(blocks).size(), "illicit share": pd.Series(lab_y).groupby(blocks).mean().round(4)}))

             rows  illicit share
days 1-22   21293         0.0953
days 23-35   9942         0.1624


## 3. Feature engineering

**The final model uses the 165 supplied features unchanged.** `feat_1`-`feat_93` describe the transaction itself,
`feat_94`-`feat_165` are the organisers' summaries of its immediate neighbours. I tried many engineered features, graph
features, feature removals and transformations. None of them improved the leaderboard score, so none are in the model.
Two small cells below show the evidence behind the two most important lessons; the table after them lists the rest.

### 3.1 Six features act as time markers

The test days (36-49) come after every training day, and a few features drift with time. A single feature can tell a
recent training row from a test row. An AUC of 0.5 means the feature carries no information about which period a row comes from.

In [6]:
def solo_auc(a, b):
    """Univariate AUC of every column for telling rows of b from rows of a (rank-sum / Mann-Whitney)."""
    ranks = rankdata(np.vstack([a, b]).astype("float64"), axis=0)
    n_a, n_b = len(a), len(b)
    return (ranks[n_a:].sum(axis=0) - n_b * (n_b + 1) / 2) / (n_a * n_b)

recent = lab_X[lab_day >= 18]                     # labelled rows of days 18-35 against the test rows
auc = pd.Series(solo_auc(recent, test_X), index=FEATURES)
low, high = lab_X.min(axis=0), lab_X.max(axis=0)  # range of every feature in the labelled training rows
outside = pd.Series(((test_X < low) | (test_X > high)).mean(axis=0), index=FEATURES)

markers = pd.DataFrame({
    "train-vs-test AUC": auc,
    "separation": (auc - 0.5).abs() + 0.5,        # 0.5 = no information, 1.0 = perfect
    "test rows outside the training range": outside,
}).sort_values("separation", ascending=False)
print(markers.head(8).round(3).to_string())
print("\nfeatures with separation above 0.90:", ", ".join(markers.index[markers["separation"] > 0.90]))
print("features with separation above 0.75 (labelled days 18-35 against test):", int((markers["separation"] > 0.75).sum()), "of 165")
auc_all = pd.Series(solo_auc(lab_X, test_X), index=FEATURES)       # the same test with every labelled day
print("features with separation above 0.75 (all labelled days against test):", int((((auc_all - 0.5).abs() + 0.5) > 0.75).sum()), "of 165")

          train-vs-test AUC  separation  test rows outside the training range
feat_101              0.998       0.998                                 0.995
feat_103              0.997       0.997                                 0.994
feat_100              0.996       0.996                                 0.992
feat_136              0.994       0.994                                 0.000
feat_139              0.953       0.953                                 0.000
feat_137              0.910       0.910                                 0.000
feat_2                0.814       0.814                                 0.001
feat_109              0.770       0.770                                 0.054

features with separation above 0.90: feat_101, feat_103, feat_100, feat_136, feat_139, feat_137
features with separation above 0.75 (labelled days 18-35 against test): 8 of 165
features with separation above 0.75 (all labelled days against test): 17 of 165


Six features separate the training period from the test period with an AUC above 0.90, and `feat_100`, `feat_101` and
`feat_103` lie outside the training range for more than 99% of test rows (progress.md sections 4.4 and 25). A model that leans on them is
extrapolating. Two ways to remove the risk were tried on the TabPFN recipe: **clipping** the test values into each bag's context
range barely changed the ranking (Spearman 0.992 and 0.995, so there was nothing to submit), and **dropping** the six features
cost 0.0042 on the leaderboard (0.96402 against 0.96825). TabPFN seems to get useful signal from them even out of range (progress.md section 25), so they stay.

### 3.2 Neighbour counts break because train and test hold different nodes

`train.csv` holds every node of days 1-35, labelled or not. `test.csv` holds only the labelled nodes of days 36-49.
Any graph feature that counts "neighbours without a feature row" therefore behaves differently in the two files.
The cell below builds that count from the edge list.

In [7]:
edges = pd.read_csv(DATA / "txs_edgelist.csv")
has_row = np.union1d(train["txId"].to_numpy(), test["txId"].to_numpy())      # nodes that have a feature row

# Every edge counts for both of its ends.
node = np.r_[edges["txId1"].to_numpy(), edges["txId2"].to_numpy()]
other = np.r_[edges["txId2"].to_numpy(), edges["txId1"].to_numpy()]
unknown = pd.Series(~np.isin(other, has_row)).groupby(node).sum()            # neighbours with no feature row
degree = pd.Series(1, index=node).groupby(level=0).sum()

def graph_stats(frame):
    ids = frame["txId"]
    return pd.Series({
        "rows": len(frame),
        "mean unknown neighbours": ids.map(unknown).fillna(0).mean(),
        "rows with at least one": (ids.map(unknown).fillna(0) > 0).mean(),
        "mean degree": ids.map(degree).fillna(0).mean(),
    })

print(pd.DataFrame({"train rows": graph_stats(train), "test rows": graph_stats(test)}).round(4).to_string())
assert (train["txId"].map(unknown).fillna(0) == 0).all(), "expected 0 unknown neighbours for every train row"

                          train rows   test rows
rows                     141772.0000  15329.0000
mean unknown neighbours       0.0000      1.5816
rows with at least one        0.0000      0.5315
mean degree                   2.3022      3.2418


The count is **exactly 0 for every training row**, but non-zero for about half of the test rows (mean 1.58 unknown neighbours).
A model trained on it sees a constant, learns nothing from it, and then meets values it has never seen at test time.
Two such features (`g_unknown_neighbours`, `g_unknown_fraction`) were part of the graph feature set that scored 0.89413 on the leaderboard
against 0.94812 before, a loss of 0.054, although it had looked better offline (progress.md sections 2 and 7). The code that built these two features was not kept; the cell above recomputes the count from the edge list.
The lesson was to check every engineered feature for train/test shift before using it, and the final model uses none.

### 3.3 Feature changes that lost on the leaderboard

All figures are public leaderboard scores from `reports/progress.md` (the last column gives its section).

| Change to the features | Score | Compared with | progress.md section |
|---|---|---|---|
| 22 engineered graph features added to the 165 | 0.89413 | 0.94812 | 2, 7 |
| Drop 21 distribution-shifted features | 0.93984 | 0.94812 | 2 |
| Replace 17 drifting features (separation above 0.75 over all labelled days) by within-day ranks | 0.94921 | 0.95521 | 11 |
| Labelled-neighbour counts added to the three window models | 0.95645 | 0.95646 (tie) | 11, 16 |
| Neighbour-only model (`feat_94`-`feat_165`) blended at .20 | 0.94869 | 0.95672 | 17 |
| Local-only model (`feat_1`-`feat_93`) blended at .30 | 0.95325 | 0.95672 | 17 |
| TabPFN recipe rebuilt without the six time markers | 0.96402 | 0.96825 | 25 |
| Exact-match re-identification counts | no gain offline (-0.0031 and +0.0002) | not submitted | 26 |
| Test features clipped into the context range | Spearman 0.992 / 0.995 with the unclipped bags | not submitted | 25 |

The only things that gained were model settings, seed averaging, blending the same model over different recent windows,
and replacing gradient boosting by pretrained tabular models (section 4 of this notebook).

## 4. The model

Both models are pretrained tabular foundation models used in-context: they are not trained on this data. A **context**
of labelled rows is given to the model, and the **query** rows are scored in one forward pass.

**Context recipe, which is also how class imbalance is handled.** Only 11.67% of labelled rows are illicit. A context holds
*all* illicit rows from a first day onward plus *seeded random licit rows* up to a fixed context size. Every rare illicit row is
always seen, the licit rows are subsampled, and the model fits on a GPU with 8 GB. (TabPFN was too slow to take all 31k rows on a CPU, progress.md section 20.)
A full-context run just takes every labelled row.

**Bags differ in three ways** and are combined by averaging percentile ranks, because AUC only needs the order and ranks stop one
bag's extreme probabilities from dominating:

* the seed, which decides the licit rows and the model's internal randomness;
* the first day: 1 uses all history, 18 uses the recent regime. Recent contexts helped TabPFN as they helped xgboost earlier (progress.md section 20);
* the context size (8k, 12k, full). The mix of sizes is itself a source of useful diversity (progress.md section 20).

**Why Kumo in the final model.** Every TabPFN variant, bag mix and partner model (xgboost, CatBoost, TabICLv2, other TabPFN weights)
stayed at or below 0.96825. Kumo Tabular is the only partner that added: its runs agree with each other (Spearman 0.98) but rank the
test rows differently from the TabPFN blend (Spearman 0.867 for the 22 runs and 11 bags used here, 0.863 for the first 12 runs). Adding it at .25 gave 0.97162,
and the weight search on the last day (progress.md section 29) gave the final .55.

The cell below shows what each recipe puts in the context, computed from the data.

In [8]:
def recipe_rows(first_day, context_rows):
    """Context size and illicit rows when all illicit rows from first_day on are kept and licit rows fill the rest."""
    pool = lab_y[lab_day >= first_day]
    n_ill, n_lic = int((pool == 1).sum()), int((pool == 0).sum())
    return n_ill + min(n_lic, context_rows - n_ill), n_ill

rows = []
for first_day, context_rows in [(1, 8000), (1, 12000), (18, 8000), (18, 12000), (18, 14380), (1, 31235)]:
    size, n_ill = recipe_rows(first_day, context_rows)
    rows.append({"first day": first_day, "context rows": size, "illicit rows": n_ill, "illicit share": n_ill / size})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("natural illicit share of all labelled rows:", round(float(lab_y.mean()), 4))

 first day  context rows  illicit rows  illicit share
         1          8000          3644          0.456
         1         12000          3644          0.304
        18          8000          2265          0.283
        18         12000          2265          0.189
        18         14380          2265          0.158
         1         31235          3644          0.117
natural illicit share of all labelled rows: 0.1167


| Part | Runs | Context rows | First day | Weight inside the part |
|---|---|---|---|---|
| TabPFN, all days | 6 bags | 8,000 | 1 | .167 |
| TabPFN, all days | 1 bag | 12,000 | 1 | .083 |
| TabPFN, days 18+ | 3 bags | 8,000 | 18 | .40 |
| TabPFN, days 18+ | 1 bag | 12,000 | 18 | .35 |
| Kumo, days 18+ | 10 seeds, plus 1 run | 8,000; 14,380 (all) | 18 | .06818 each (.75 in total) |
| Kumo, all days | 10 seeds, plus 1 run | 12,000; 31,235 (all) | 1 | .02273 each (.25 in total) |

The final prediction is the TabPFN blend times .45 plus the Kumo blend times .55, each as percentile ranks.

### 4.1 TabPFN bag

The code below is the recipe of `src/model_tabpfn.py` with its default settings: the package default weights (TabPFN v3.5),
2 internal estimators (the ensemble size; 8 per bag scored 0.96694 against 0.96825 on the leaderboard, README), `random_state` equal to the
bag number, `ignore_pretraining_limits=True` (the switch that lets TabPFN accept contexts above its documented pretraining sizes), the
context drawn by `np.random.default_rng(bag)`, and 2,000 query rows per call.
It takes an optional `max_day`, so the same function serves the test predictions (all labelled days) and the validation holdouts (days up to `max_day`).

In [9]:
def fresh_parts(folder, config):
    """Folder for saved chunks. It is emptied when the config changes, so stale chunks are never reused."""
    folder = Path(folder)
    meta = folder / "config.json"
    if folder.exists() and (not meta.exists() or json.loads(meta.read_text()) != config):
        for old in folder.glob("*"):
            old.unlink()
    folder.mkdir(parents=True, exist_ok=True)
    meta.write_text(json.dumps(config))
    return folder


def tabpfn_bag(first_day, context_rows, bag, Xq, max_day=999, parts_dir=None, chunk=2000):
    """One TabPFN v3.5 bag: P(illicit) for every row of Xq. Same recipe as src/model_tabpfn.py."""
    from tabpfn import TabPFNClassifier

    # Context pool: labelled rows from first_day to max_day.
    keep = (lab_day >= first_day) & (lab_day <= max_day)
    X, y = lab_X[keep], lab_y[keep]
    illicit, licit = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    # All illicit rows, plus a seeded random draw of licit rows up to the context size (class imbalance).
    rng = np.random.default_rng(bag)
    take = min(len(licit), context_rows - len(illicit))
    rows = np.r_[illicit, rng.choice(licit, take, replace=False)]

    model = TabPFNClassifier(device="cuda", n_estimators=2, random_state=bag, ignore_pretraining_limits=True)
    model.fit(X[rows], y[rows])

    if parts_dir is not None:     # every chunk is saved, so a stopped run resumes where it left off
        parts_dir = fresh_parts(parts_dir, dict(first_day=first_day, max_day=max_day, context=context_rows, bag=bag,
                                                n_query=len(Xq), check=float(Xq[:, 0].astype("float64").sum())))
    out = []
    for i in range(0, len(Xq), chunk):
        saved = None if parts_dir is None else parts_dir / f"chunk_{i}.npy"
        if saved is not None and saved.exists():
            out.append(np.load(saved))
            continue
        out.append(model.predict_proba(Xq[i:i + chunk])[:, 1])
        if saved is not None:
            np.save(saved, out[-1])

    del model                     # free the GPU before the next job
    gc.collect()
    torch.cuda.empty_cache()
    return np.concatenate(out)

### 4.2 Kumo Tabular runs

Kumo Tabular needs a newer PyTorch (2.11, CUDA 12.8) than TabPFN (2.5.1, CUDA 12.4), so it cannot run in this kernel.
The cell below writes a small script to `cache/notebook/kumo_runner.py`, and the notebook runs it with the second environment
(`.venv-fm`) through `subprocess`. The script is the recipe of `src/fm/fm_common.py` and `src/fm/run_kumo.py`: model size `large`,
8 estimators, 1,000 query rows per call, the same context sampling and random-number order, the cap on the illicit share of the context
(0.5), a shuffled context, fp16 autocast (to save time and memory on the 8 GB card), and the illicit probability selected by class name `"1"` because the class order is shuffled in every call.

`src/fm/run_kumo.py` is the original; this script is a self-contained copy of the same recipe. Its context arrays equal those of
`src/fm/fm_common.py` for all 22 configurations, and it reproduces the committed `kumo_recent18_s0.csv` and `kumo_allfull_s0.csv` byte for byte.

It reads the labelled arrays and a query matrix from disk and takes `--max-day`, so test mode and validation mode go through the
same code: the test mode scores `test_X.npy`, the validation mode scores a holdout of labelled days while the context stops at `--max-day`.

In [10]:
%%writefile cache/notebook/kumo_runner.py
"""Kumo Tabular runner for e1122376.ipynb. Run it with .venv-fm (torch 2.11, structured-data-models 0.2.1).

Same recipe as src/fm/fm_common.py + src/fm/run_kumo.py: the context is ALL illicit rows from --first-day on (capped at
--max-illicit-frac of the context) plus seeded random licit rows, shuffled; Kumo Tabular large, 8 estimators.
--max-day stops the context at a day, which is used for validation holdouts. Writes index,target.
"""
import argparse
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
FEATURES = [f"feat_{i}" for i in range(1, 166)]


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def build_context(X, y, day, first_day, max_day, context_rows, seed, max_illicit_frac=0.5):
    """All illicit rows (capped) + seeded random licit rows from days first_day..max_day, shuffled."""
    usable = (day >= first_day) & (day <= max_day)
    ill = np.flatnonzero(usable & (y == 1))
    lic = np.flatnonzero(usable & (y == 0))
    rng = np.random.default_rng(seed)
    max_ill = int(max_illicit_frac * context_rows)
    if len(ill) > max_ill:      # only bites when there are very many illicit rows
        ill = ill[np.sort(rng.choice(len(ill), size=max_ill, replace=False))]
    n_lic = min(len(lic), max(context_rows - len(ill), 0))
    lic = lic[np.sort(rng.choice(len(lic), size=n_lic, replace=False))]
    rows = np.r_[ill, lic]
    order = pd.DataFrame(index=np.arange(len(rows))).sample(frac=1.0, random_state=seed).index.to_numpy()
    rows = rows[order]          # shuffle the context rows
    return X[rows].astype(np.float32), y[rows].astype(np.int64)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)        # labelled.npz with X, y, day
    ap.add_argument("--query", required=True)       # .npy with the rows to score
    ap.add_argument("--index", default="")          # .npy with the id of every query row (default 0..n-1)
    ap.add_argument("--out", required=True)
    ap.add_argument("--first-day", type=int, required=True)
    ap.add_argument("--max-day", type=int, default=999)
    ap.add_argument("--context-rows", type=int, required=True)
    ap.add_argument("--seed", type=int, required=True)
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--chunk", type=int, default=1000)
    ap.add_argument("--size", default="large")
    ap.add_argument("--estimators", type=int, default=8)
    ap.add_argument("--max-illicit-frac", type=float, default=0.5)
    ap.add_argument("--max-query", type=int, default=0)   # smoke tests: score only the first N query rows
    args = ap.parse_args()

    import torch
    import sdm

    dev = torch.device(args.device)
    log(f"Kumo Tabular size={args.size} estimators={args.estimators} device={dev} torch={torch.__version__} sdm={sdm.__version__}")

    data = np.load(args.data)
    xc, yc = build_context(data["X"], data["y"], data["day"], args.first_day, args.max_day,
                           args.context_rows, args.seed, args.max_illicit_frac)
    xt = np.load(args.query).astype(np.float32)
    index = np.load(args.index) if args.index else np.arange(len(xt))
    if args.max_query:
        xt, index = xt[: args.max_query], index[: args.max_query]
    log(f"context: {len(xc)} rows ({int((yc == 1).sum())} illicit), days {args.first_day}-{args.max_day}; query: {len(xt)} rows")

    ctx_df = pd.DataFrame(xc, columns=FEATURES)
    ctx_df["target"] = yc
    stypes = {c: sdm.Stype.numerical for c in FEATURES}
    ctx_tab = sdm.TableTensor.from_pandas(df=ctx_df, stypes={**stypes, "target": sdm.Stype.categorical}, device=dev)
    x_ctx = ctx_tab.drop_columns("target")
    y_ctx = ctx_tab[:, "target"]

    model = sdm.models.KumoTabular(task="classification", size=args.size, device=dev)
    torch.manual_seed(args.seed)
    gen = torch.Generator(device=dev).manual_seed(args.seed)
    amp = torch.amp.autocast(dev.type, dtype=torch.float16, enabled=dev.type == "cuda")

    def p_illicit(out):
        names = list(out.columns[sdm.Stype.numerical])      # class order is shuffled per call -> select by name
        return out.numerical.float().cpu().numpy()[:, names.index("1")]

    # Saved chunks let a stopped run resume. They are discarded if anything about the run changed.
    out_csv = Path(args.out)
    parts = out_csv.with_name(out_csv.name + ".parts")
    config = dict(vars(args), n_query=len(xt), check=float(xt[:, 0].astype("float64").sum()))
    meta = parts / "config.json"
    if parts.exists() and (not meta.exists() or json.loads(meta.read_text()) != config):
        for old in parts.glob("*"):
            old.unlink()
    parts.mkdir(parents=True, exist_ok=True)
    meta.write_text(json.dumps(config))

    preds = []
    with torch.no_grad(), amp:
        model.fit(x=x_ctx, y=y_ctx, num_estimators=args.estimators, generator=gen)   # caches the context once
        for s in range(0, len(xt), args.chunk):
            saved = parts / f"part_{s}.npy"
            if saved.exists():
                preds.append(np.load(saved))
                continue
            tab = sdm.TableTensor.from_pandas(df=pd.DataFrame(xt[s: s + args.chunk], columns=FEATURES), stypes=stypes, device=dev)
            p = np.asarray(p_illicit(model.predict(tab)), dtype=np.float64)
            np.save(saved, p)
            preds.append(p)
            log(f"chunk {s}-{min(s + args.chunk, len(xt))} done")
    p = np.concatenate(preds)
    assert p.shape == (len(xt),) and np.isfinite(p).all()
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"index": index, "target": p}).to_csv(out_csv, index=False)
    log(f"wrote {out_csv.name}: {len(p)} rows, mean P(illicit)={p.mean():.4f}")


if __name__ == "__main__":
    main()

Overwriting cache/notebook/kumo_runner.py


In [11]:
def run_kumo(first_day, context_rows, seed, query_file, out_csv, max_day=999, index_file=None, max_query=0):
    """Run one Kumo Tabular job in .venv-fm and return P(illicit) for the query rows."""
    if not KUMO_PY.exists():
        raise FileNotFoundError(f"{KUMO_PY} not found: create .venv-fm as described in requirements-fm.txt")
    cmd = [str(KUMO_PY), str(WORK / "kumo_runner.py"), "--data", str(WORK / "labelled.npz"), "--query", str(query_file),
           "--out", str(out_csv), "--first-day", str(first_day), "--max-day", str(max_day),
           "--context-rows", str(context_rows), "--seed", str(seed)]
    if index_file is not None:
        cmd += ["--index", str(index_file)]
    if max_query:
        cmd += ["--max-query", str(max_query)]
    env = {k: v for k, v in os.environ.items() if k != "TABPFN_TOKEN"}   # the child does not need the token
    env["PYTHONIOENCODING"] = "utf-8"
    start = time.time()
    done = subprocess.run(cmd, capture_output=True, text=True, env=env, cwd=REPO)
    if done.returncode != 0:
        print(done.stdout[-3000:], done.stderr[-3000:])
        raise RuntimeError(f"Kumo run failed for {Path(out_csv).name}")
    print(f"  {Path(out_csv).name}: {time.time() - start:.0f}s | {done.stdout.strip().splitlines()[-1]}")
    return pd.read_csv(out_csv)["target"].to_numpy()


def blend_arrays(parts):
    """Rank-average arrays: sum of weight * percentile rank, divided by the sum of the weights."""
    total = sum(float(w) for _, w in parts)
    return sum(pd.Series(p).rank(pct=True).to_numpy() * float(w) for p, w in parts) / total


def rank_blend(parts, folder):
    """Rank-average prediction files in a folder. Same arithmetic as src/blend.py."""
    total, blended = 0.0, None
    for name, weight in parts:
        frame = pd.read_csv(Path(folder) / name)
        ranks = frame["target"].rank(pct=True) * float(weight)
        blended = ranks if blended is None else blended + ranks
        total += float(weight)
    return pd.DataFrame({"index": frame["index"], "target": blended / total})

### 4.3 Pretrained checkpoints and the "no external data" rule

The competition allows only the provided training and test data. The final model reads no other rows or labels, but it does start from two
sets of **pretrained weights**, which were trained by their authors on data that is not part of this competition. I treat the weights as part of
the model, like a library, and state it here so the reader can judge.

| Model | Source | Version | Access |
|---|---|---|---|
| TabPFN | Prior Labs, package `tabpfn` 9.2.0, default checkpoint | v3.5 (`TabPFNV3p5`, printed below) | downloaded on first use with a Prior Labs account token (`TABPFN_TOKEN`) |
| Kumo Tabular `large` | Hugging Face `nvidia/Kumo-Tabular`, package `structured-data-models` 0.2.1 | revision `v1.0.1` (fixed inside the package) | downloaded on first use, not gated, OpenMDW 1.1 licence |

Both downloads need internet access the first time. The committed TabPFN bags were identified as v3.5 by a rebuild check (Spearman 0.997,
progress.md section 22.1), not by a stored version string.

**Literature reading.** `reports/progress.md` section 18 records a short reading of published results on the same day-based split. It found nothing that
beat the approach already used and changed nothing in the model. It is mentioned here so that the "no outside data" statement is complete.


In [12]:
from tabpfn import TabPFNClassifier

# Load the default TabPFN checkpoint on a 200-row toy fit, only to show which model the package loads.
toy = np.r_[np.flatnonzero(lab_y == 1)[:100], np.flatnonzero(lab_y == 0)[:100]]
probe = TabPFNClassifier(device="cuda" if torch.cuda.is_available() else "cpu", n_estimators=1).fit(lab_X[toy], lab_y[toy])
print("TabPFN package", version("tabpfn"), "| default checkpoint loaded:", type(probe.model_).__name__)
del probe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if KUMO_PY.exists():
    sdm_version = subprocess.run([str(KUMO_PY), "-c", "import sdm; print(sdm.__version__)"], capture_output=True, text=True).stdout.strip()
    print("Kumo Tabular: structured-data-models", sdm_version, "| checkpoint nvidia/Kumo-Tabular, revision v1.0.1, size 'large'")


TabPFN package 9.2.0 | default checkpoint loaded: TabPFNV3p5
Kumo Tabular: structured-data-models 0.2.1 | checkpoint nvidia/Kumo-Tabular, revision v1.0.1, size 'large'


## 5. Prediction validation

The test set is in the future, so validation has to be a **time-based holdout**: the context only contains days before the
days being scored, and rows are never shuffled across time. The models are run live here, with the same functions and settings
as the test predictions (one bag or seed per component instead of all of them, to keep the GPU time to about 10 minutes).

* **Window A** is the same regime as the test recipe: context from labelled days 1-29, score labelled days 30-35.
  It checks every component and the blend with the final weights (TabPFN part: all-days .25 and days-18+ .75;
  Kumo part: days-18+ .75 and all-days .25; final: TabPFN .45 and Kumo .55).
* **Window B** crosses the day-23 regime break (progress.md section 4.1): context days 1-21, score days 22-35, all-days components only.
  The same break is why the project's earlier time folds could not predict the leaderboard. The pretrained models run with one bag or seed here.
* A **tuned xgboost** (settings of `src/model_tuned.py`: depth 12, `scale_pos_weight` 7.57, `min_child_weight` 8, 79 trees, 5 seeds rank-averaged)
  is trained on the same days as the reference. It was the model of record before TabPFN (public 0.95521).

The AUC is pooled over the scored days, as on the leaderboard. **Window A is not an independent holdout.** The design choices it checks
(TabPFN over xgboost, days-18+ contexts, the context sizes) were first motivated by evaluations on these same late days (progress.md sections 4.3
and 20), so it overstates how well the model does on the test days: about 0.9985 here against 0.97357 on the public leaderboard.

Results are cached in `VALID` (see section 1), so a second execution is quick. `E1122376_SKIP_VALIDATION=1` skips this section.


In [13]:
WINDOWS = {
    "A": dict(ctx_end=29, score=(30, 35)),    # same regime as the test recipe
    "B": dict(ctx_end=21, score=(22, 35)),    # across the day-23 regime break
}
for key, w in WINDOWS.items():
    rows = (lab_day >= w["score"][0]) & (lab_day <= w["score"][1])
    w["Xq"], w["yq"] = lab_X[rows], lab_y[rows]
    np.save(WORK / f"val{key}_X.npy", w["Xq"])             # the Kumo script reads the query from disk
    ctx = lab_day <= w["ctx_end"]
    print(f"window {key}: context = labelled days 1-{w['ctx_end']} ({int(ctx.sum()):,} rows, {int(lab_y[ctx].sum()):,} illicit);"
          f" scored = days {w['score'][0]}-{w['score'][1]} ({len(w['yq']):,} rows, {int(w['yq'].sum()):,} illicit)")


def cached_prediction(name, config, compute):
    """Run compute() once and keep the result in cache/notebook/validation/, so re-running the notebook is quick."""
    npy, meta = VALID / f"{name}.npy", VALID / f"{name}.json"
    if npy.exists() and meta.exists() and json.loads(meta.read_text()) == config:
        print(f"  loaded {name} from the cache")
        return np.load(npy)
    assert torch.cuda.is_available(), "validation needs the GPU (set E1122376_SKIP_VALIDATION=1 to skip section 5)"
    start = time.time()
    pred = np.asarray(compute(), dtype=np.float64)
    np.save(npy, pred)
    meta.write_text(json.dumps(config))
    print(f"  computed {name} in {time.time() - start:.0f}s")
    return pred


XGB_PARAMS = dict(max_depth=12, colsample_bytree=0.7, subsample=1.0, learning_rate=0.1, scale_pos_weight=7.57,
                  min_child_weight=8, gamma=0.2, n_estimators=79, tree_method="hist", eval_metric="auc", n_jobs=-1)

def xgboost_reference(max_day, Xq, seeds=(0, 1, 2, 3, 4)):
    """Tuned xgboost on labelled days <= max_day; the rank average of 5 seeds (src/model_tuned.py, src/train.py)."""
    keep = lab_day <= max_day
    score = np.zeros(len(Xq))
    for seed in seeds:
        model = XGBClassifier(**XGB_PARAMS, random_state=seed).fit(lab_X[keep], lab_y[keep])
        score += pd.Series(model.predict_proba(Xq)[:, 1]).rank(pct=True).to_numpy()
    return score / len(seeds)

window A: context = labelled days 1-29 (26,381 rows, 2,871 illicit); scored = days 30-35 (4,854 rows, 773 illicit)
window B: context = labelled days 1-21 (19,530 rows, 1,871 illicit); scored = days 22-35 (11,705 rows, 1,773 illicit)


The GPU jobs run one after another (a TabPFN bag in this kernel, then a Kumo job in a separate process), never at the same time.

In [14]:
if SKIP_VALIDATION:
    print("section 5 skipped (E1122376_SKIP_VALIDATION=1)")
else:
    preds = {}                                      # (window, model) -> scores for the scored rows

    for key, w in WINDOWS.items():
        end, Xq = w["ctx_end"], w["Xq"]
        cfg = lambda **k: dict(window=key, ctx_end=end, n_query=len(Xq), **k)
        qfile = WORK / f"val{key}_X.npy"

        # Models trained on the all-days context (first day 1): both windows.
        preds[key, "TabPFN all-days 8k"] = cached_prediction(
            f"{key}_tabpfn_all8k_b0", cfg(kind="tabpfn", first_day=1, ctx=8000, bag=0),
            lambda: tabpfn_bag(1, 8000, 0, Xq, max_day=end))
        preds[key, "Kumo all-days 12k"] = cached_prediction(
            f"{key}_kumo_all12k_s0", cfg(kind="kumo", first_day=1, ctx=12000, seed=0),
            lambda: run_kumo(1, 12000, 0, qfile, VALID / f"{key}_kumo_all12k_s0.csv", max_day=end))

        if key == "A":                               # days-18+ components only in window A (window B ends at day 21)
            preds[key, "TabPFN days-18+ 8k"] = cached_prediction(
                f"{key}_tabpfn_recent8k_b0", cfg(kind="tabpfn", first_day=18, ctx=8000, bag=0),
                lambda: tabpfn_bag(18, 8000, 0, Xq, max_day=end))
            preds[key, "Kumo days-18+ 8k"] = cached_prediction(
                f"{key}_kumo_recent8k_s0", cfg(kind="kumo", first_day=18, ctx=8000, seed=0),
                lambda: run_kumo(18, 8000, 0, qfile, VALID / f"{key}_kumo_recent8k_s0.csv", max_day=end))

        preds[key, "xgboost (tuned, 5 seeds)"] = cached_prediction(
            f"{key}_xgb", cfg(kind="xgb", params=XGB_PARAMS, seeds=5), lambda: xgboost_reference(end, Xq))

    # Blends with the final weights.
    p = preds
    p["A", "TabPFN part (all .25 + recent .75)"] = blend_arrays([(p["A", "TabPFN all-days 8k"], "0.25"), (p["A", "TabPFN days-18+ 8k"], "0.75")])
    p["A", "Kumo part (recent .75 + all .25)"] = blend_arrays([(p["A", "Kumo days-18+ 8k"], "0.75"), (p["A", "Kumo all-days 12k"], "0.25")])
    p["A", "FINAL BLEND (TabPFN .45 + Kumo .55)"] = blend_arrays([(p["A", "TabPFN part (all .25 + recent .75)"], "0.45"), (p["A", "Kumo part (recent .75 + all .25)"], "0.55")])
    p["B", "FINAL BLEND weights, all-days parts only"] = blend_arrays([(p["B", "TabPFN all-days 8k"], "0.45"), (p["B", "Kumo all-days 12k"], "0.55")])


  loaded A_tabpfn_all8k_b0 from the cache
  loaded A_kumo_all12k_s0 from the cache
  loaded A_tabpfn_recent8k_b0 from the cache
  loaded A_kumo_recent8k_s0 from the cache
  loaded A_xgb from the cache
  loaded B_tabpfn_all8k_b0 from the cache
  loaded B_kumo_all12k_s0 from the cache
  loaded B_xgb from the cache


In [15]:
if SKIP_VALIDATION:
    print("section 5 skipped (E1122376_SKIP_VALIDATION=1)")
else:
    validation = pd.DataFrame(
        [{"window": f"{k} (context days 1-{WINDOWS[k]['ctx_end']}, score days {WINDOWS[k]['score'][0]}-{WINDOWS[k]['score'][1]})",
          "model": m, "ROC AUC": roc_auc_score(WINDOWS[k]["yq"], s)} for (k, m), s in preds.items()]
    ).sort_values(["window", "ROC AUC"], ascending=[True, False])
    validation["ROC AUC"] = validation["ROC AUC"].round(5)
    print(validation.to_string(index=False))

    for key in WINDOWS:
        a, b = preds[key, "TabPFN all-days 8k"], preds[key, "Kumo all-days 12k"]
        print(f"window {key}: Spearman between the all-days TabPFN and Kumo scores = {spearmanr(a, b).correlation:.3f}")


                                 window                                    model  ROC AUC
A (context days 1-29, score days 30-35)      FINAL BLEND (TabPFN .45 + Kumo .55)  0.99854
A (context days 1-29, score days 30-35)       TabPFN part (all .25 + recent .75)  0.99827
A (context days 1-29, score days 30-35)         Kumo part (recent .75 + all .25)  0.99821
A (context days 1-29, score days 30-35)                       TabPFN days-18+ 8k  0.99819
A (context days 1-29, score days 30-35)                         Kumo days-18+ 8k  0.99818
A (context days 1-29, score days 30-35)                        Kumo all-days 12k  0.99792
A (context days 1-29, score days 30-35)                       TabPFN all-days 8k  0.99791
A (context days 1-29, score days 30-35)                 xgboost (tuned, 5 seeds)  0.99713
B (context days 1-21, score days 22-35)                 xgboost (tuned, 5 seeds)  0.91387
B (context days 1-21, score days 22-35)                       TabPFN all-days 8k  0.90549
B (context

**Reading the validation** (numbers from the executed run; a re-run on the GPU can differ in the last digits).

* **Window A** is the closer stand-in for the test task, because its context already contains days after the break.
  Every single component scores about 0.998, blending the runs of a part helps a little, and the final blend is the best model
  (0.99854), ahead of the tuned xgboost (0.99713). The gaps are small because an offline window past the break has little
  headroom: eight very different xgboost variants spread over only 0.0012 AUC on days 25-35 (progress.md section 4.2). So window A shows that
  every component works and that the blend does not hurt, but it cannot rank close alternatives, and it cannot choose the weights.
* **Window B** crosses the break and its context stops at day 21. Here the order flips: xgboost 0.914, TabPFN 0.905, the
  blend 0.888, Kumo 0.867. This window does **not** support the final weights, and I report it as it is. The project found earlier that
  windows ending before the break do not predict the leaderboard: five validation protocols, including pre-break ones, all ranked the known
  leaderboard loser first (progress.md section 4.3), and the test days behave like the post-break regime (progress.md section 17). A context that stops
  before the break cannot show the later regime to any model; a possible reason why the in-context models lose most here is that they
  follow the regime in their context, but I did not test that.

The weights therefore come from the **public leaderboard (30% of the test rows)**, which was the main judge, not from either window.

Public leaderboard scores of the models of record:

| Model | Public score |
|---|---|
| Organisers' baseline | 0.92119 |
| Tuned xgboost (`src/model_tuned.py`) | 0.95521 |
| Best TabPFN-only blend | 0.96825 |
| **TabPFN .45 + Kumo .55 (this notebook)** | **0.97357** |

The Kumo weight was searched on the leaderboard on the last day. The TabPFN part was the same in all four, but the .25 point used the
first, 12-run Kumo part and the other three used the final 22-run Kumo part (the two Kumo parts have Spearman 0.9995):

| Kumo weight | Public score |
|---|---|
| .25 (12-run Kumo part) | 0.97162 |
| .33 (22-run Kumo part) | 0.97229 |
| .40 | 0.97280 |
| .55 | **0.97357** |

The four points rise smoothly. One caveat from progress.md section 13: the public score has a noise of about 0.001, so these are not
precise optima. The choice of .55 is the best of the four tested, and no larger weight was tried. The weights were tuned on the public 30% of the test rows,
so they may not transfer exactly to the private 70%.

## 6. Test predictions and the submission file

Section 6 produces the per-run files (`SRC` folder), blends them and writes the submission.

* **Fast path:** `SRC` is `submissions/`, which holds the committed per-run files (4 TabPFN files and 22 Kumo files). They were made earlier by
  `src/model_tabpfn.py` and `src/fm/run_kumo.py`; this notebook only re-blends them.
* **Full rerun:** every TabPFN bag and Kumo run is recomputed from the raw data into `rebuild/rerun/`. The saved bags in `cache/`
  and the files in `submissions/` are never read. Runs that already finished are skipped, so an interrupted rerun resumes
  (after an abort, check `nvidia-smi` for a leftover Kumo process before resuming). Delete `rebuild/rerun/` whenever the recipe or the inputs change,
  because finished files are not re-checked. **GPU runs are not bitwise identical to the committed bags** (several were run on CPU and the
  GPU kernels are not deterministic; a GPU rebuild of the days-18+ bags correlates 0.997 with them, and one bag run on two devices
  correlates 0.989), so a rerun is compared with Spearman correlation, not by equality. Expected: about 0.98 or more for the single-bag TabPFN
  files, about 0.997 for the multi-bag ones, and at least 0.999 for the Kumo files (the Kumo runs on this GPU were bit-identical).

A TabPFN file with several bags is the mean percentile rank of its bags. The two single-bag files (`tabpfn_ctx12k_bag0.csv`,
`tabpfn_recent18ctx12k_bag0.csv`) hold the raw P(illicit) of their bag, as committed; the blend re-ranks every file, so this changes nothing.
Files are written as `index,target` in `test.csv` order. Full rerun timing: about 45 minutes of TabPFN and 1.5 to 2 hours of Kumo (the 31k-row run alone takes about 28 minutes).


In [16]:
SRC = OUT if RERUN else SUB
start_all = time.time()

if RERUN:
    assert torch.cuda.is_available(), "a full rerun needs the GPU"
    # --- TabPFN bags (this kernel) ---
    for name, first_day, context, bags, _ in TABPFN_RUNS:
        if (OUT / name).exists():
            print("skip (done):", name)
            continue
        t0, ranks = time.time(), 0
        for bag in bags:
            raw = tabpfn_bag(first_day, context, bag, test_X, parts_dir=OUT / "parts" / f"{name}.bag{bag}")
            ranks = ranks + pd.Series(raw).rank(pct=True).to_numpy()      # a file = the mean of its bags' percentile ranks
            print(f"  {name} bag {bag} done ({time.time() - t0:.0f}s so far)")
        # as committed: several bags -> the mean percentile rank; a single bag -> its raw P(illicit)
        pd.DataFrame({"index": test_index, "target": ranks / len(bags) if len(bags) > 1 else raw}).to_csv(OUT / name, index=False)
    # --- Kumo runs (.venv-fm, one process per run) ---
    for name, first_day, context, seed, _ in KUMO_RUNS:
        if (OUT / name).exists():
            print("skip (done):", name)
            continue
        run_kumo(first_day, context, seed, WORK / "test_X.npy", OUT / name, index_file=WORK / "test_index.npy")
    print(f"all runs finished in {(time.time() - start_all) / 60:.1f} minutes")

# Every per-run file must list the rows in submission order and hold finite numbers.
for name in [r[0] for r in TABPFN_RUNS] + [r[0] for r in KUMO_RUNS]:
    frame = pd.read_csv(SRC / name)
    assert len(frame) == 15329 and (frame["index"].to_numpy() == sample["index"].to_numpy()).all(), name
    assert np.isfinite(frame["target"]).all(), name
print(f"all {len(TABPFN_RUNS) + len(KUMO_RUNS)} per-run files checked in {SRC.relative_to(REPO)}")

  tabpfn_all_bags0-5.csv bag 0 done (177s so far)
  tabpfn_all_bags0-5.csv bag 1 done (353s so far)
  tabpfn_all_bags0-5.csv bag 2 done (553s so far)
  tabpfn_all_bags0-5.csv bag 3 done (732s so far)
  tabpfn_all_bags0-5.csv bag 4 done (904s so far)
  tabpfn_all_bags0-5.csv bag 5 done (1074s so far)
  tabpfn_ctx12k_bag0.csv bag 0 done (318s so far)
  tabpfn_recent18_bags0-2.csv bag 0 done (153s so far)
  tabpfn_recent18_bags0-2.csv bag 1 done (328s so far)
  tabpfn_recent18_bags0-2.csv bag 2 done (487s so far)
  tabpfn_recent18ctx12k_bag0.csv bag 0 done (311s so far)
  kumo_recent18_s0.csv: 138s | 15:31:50 wrote kumo_recent18_s0.csv: 15329 rows, mean P(illicit)=0.0669
  kumo_recent18_s1.csv: 128s | 15:33:58 wrote kumo_recent18_s1.csv: 15329 rows, mean P(illicit)=0.0658
  kumo_recent18_s2.csv: 128s | 15:36:07 wrote kumo_recent18_s2.csv: 15329 rows, mean P(illicit)=0.0628
  kumo_recent18_s3.csv: 127s | 15:38:14 wrote kumo_recent18_s3.csv: 15329 rows, mean P(illicit)=0.0642
  kumo_recent1

**Blending.** The blends are rank averages, in three steps (README, "Reproducing the submission", steps 2 and 4):
the TabPFN blend from the 4 TabPFN files, the Kumo blend from the 22 Kumo runs, and the final blend of those two with weights .45 and .55.
The intermediate blends are written to disk and read back, as `src/blend.py` does, so the arithmetic is identical.

In [17]:
tabpfn_blend = rank_blend([(r[0], r[4]) for r in TABPFN_RUNS], SRC)
tabpfn_blend.to_csv(OUT / "blend_tabpfn_recent75_12k.csv", index=False)

kumo_blend = rank_blend([(r[0], r[4]) for r in KUMO_RUNS], SRC)
kumo_blend.to_csv(OUT / "kumo22_blend.csv", index=False)

final = rank_blend(FINAL_PARTS, OUT)                    # reads the two blends written just above
final.to_csv(OUT / "blend_tabpfn_kumo22_55.csv", index=False)
final.to_csv(OUT / "e1122376_submission.csv", index=False)    # the file to submit to Kaggle

print("submission written:", (OUT / "e1122376_submission.csv").relative_to(REPO), "|", final.shape)
print("Spearman, TabPFN blend vs Kumo blend:", round(spearmanr(tabpfn_blend["target"], kumo_blend["target"]).correlation, 3))
print(final.head().to_string(index=False))

submission written: rebuild\rerun\e1122376_submission.csv | (15329, 2)
Spearman, TabPFN blend vs Kumo blend: 0.872
 index   target
     0 0.531323
     1 0.630641
     2 0.630834
     3 0.540247
     4 0.216713


## 7. Checks

**Rebuilt against committed.** The committed files in `submissions/` were produced by `src/blend.py` from the same inputs.
The comparison reads both sides with `float_precision="round_trip"`, so a number is compared as written in the file.
In the fast path the three blends must be identical (largest absolute difference 0). In a full rerun the per-run files
come from new GPU runs, so the Spearman correlation and the largest difference are reported, and any file below the expected agreement
(section 6) is listed. Note that the fast path compares blends of the same input files, so it checks the arithmetic only.

In [18]:
def read_exact(path):
    return pd.read_csv(path, float_precision="round_trip")      # no 1-ulp parsing differences

names = ["blend_tabpfn_recent75_12k.csv", "kumo22_blend.csv", "blend_tabpfn_kumo22_55.csv"]
if RERUN:
    names = [r[0] for r in TABPFN_RUNS] + [r[0] for r in KUMO_RUNS] + names

rows = []
for name in names:
    new, old = read_exact(OUT / name), read_exact(SUB / name)
    assert (new["index"].to_numpy() == old["index"].to_numpy()).all(), name
    a, b = new["target"].to_numpy(), old["target"].to_numpy()
    rows.append({"file": name, "identical": bool(np.array_equal(a, b)), "max abs diff": float(np.abs(a - b).max()),
                 "Spearman": spearmanr(a, b).correlation})
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

if not RERUN:
    # The fast path must reproduce the committed blends exactly.
    assert comparison["identical"].all(), "rebuilt blends differ from the committed ones"
    print("\nOK: the three rebuilt blends are identical to the committed files (max abs diff 0).")
else:
    expected = {"tabpfn": 0.98, "kumo_": 0.999}          # minimum Spearman per kind of file; blends use 0.98
    low = [r["file"] for _, r in comparison.iterrows()
           if r["Spearman"] < next((v for k, v in expected.items() if r["file"].startswith(k)), 0.98)]
    last = comparison.iloc[-1]
    print(f"\nFinal blend, rerun vs committed: Spearman {last['Spearman']:.4f}, max abs diff {last['max abs diff']:.4f}.")
    print("WARNING: below the expected agreement:" if low else "OK: every file agrees as well as device noise allows.", ", ".join(low))


                          file  identical  max abs diff  Spearman
        tabpfn_all_bags0-5.csv      False      0.071406  0.998974
        tabpfn_ctx12k_bag0.csv      False      0.379889  0.992145
   tabpfn_recent18_bags0-2.csv      False      0.145911  0.996893
tabpfn_recent18ctx12k_bag0.csv      False      0.342278  0.988786
          kumo_recent18_s0.csv       True      0.000000  1.000000
          kumo_recent18_s1.csv       True      0.000000  1.000000
          kumo_recent18_s2.csv       True      0.000000  1.000000
          kumo_recent18_s3.csv       True      0.000000  1.000000
          kumo_recent18_s4.csv       True      0.000000  1.000000
          kumo_recent18_s5.csv       True      0.000000  1.000000
          kumo_recent18_s6.csv       True      0.000000  1.000000
          kumo_recent18_s7.csv       True      0.000000  1.000000
          kumo_recent18_s8.csv       True      0.000000  1.000000
          kumo_recent18_s9.csv       True      0.000000  1.000000
      kumo

**Submission format.** The competition wants `index,target` for 15,329 rows in the order of `sample_submission.csv`, with
a risk score in [0, 1] in the `target` column. It is the weighted mean percentile rank, not a calibrated probability; the AUC only uses the order.

In [19]:
submission = pd.read_csv(OUT / "e1122376_submission.csv")
assert list(submission.columns) == ["index", "target"]
assert len(submission) == 15329 == len(sample)
assert (submission["index"].to_numpy() == sample["index"].to_numpy()).all()      # same order as sample_submission.csv
assert submission["index"].is_unique
assert not submission["target"].isna().any()
assert submission["target"].between(0, 1).all()
print("submission format OK:", (OUT / "e1122376_submission.csv").relative_to(REPO))
print(submission["target"].describe().round(4).to_string())

# Upload (not run here):  kaggle competitions submit -c bt-4012-competition-2026 -f <this file> -m "e1122376 notebook"

submission format OK: rebuild\rerun\e1122376_submission.csv
count    15329.0000
mean         0.5000
std          0.2794
min          0.0041
25%          0.2647
50%          0.4909
75%          0.7419
max          0.9999


## Summary

* The model uses the 165 supplied features unchanged: every engineered-feature, feature-removal or graph-feature variant that was submitted lost or tied
  on the leaderboard, and two more ideas gained nothing offline (section 3).
* The final prediction is the rank average of TabPFN v3.5 bags (.45) and Kumo Tabular runs (.55). Each run reads all illicit rows plus a seeded
  random draw of licit rows as context (section 4).
* Time-based holdouts (section 5): in window A (context past the day-23 break) the final blend scores best; in window B (context before the break)
  it scores below xgboost and TabPFN alone. Offline windows did not predict the leaderboard in this project, which was the main judge; window A is not independent of earlier model choices.
* In the default mode the notebook re-blends the saved per-run files, so its exact match checks the arithmetic and the file format, not the models;
  only the full rerun regenerates the predictions (not bit-identical, rank correlation reported).
* Section 6 writes `e1122376_submission.csv`, and section 7 shows that it equals the committed `submissions/blend_tabpfn_kumo22_55.csv`
  (fast path) or reports how closely a fresh GPU rerun matches it.